In [1]:
# Quantifi Toolkit imports needed to price a CDX.NA.IG Series 46 index option

from Quantifi.Toolkit.Base import (
    Dt,
    Tenor,
    Calendar,
    CycleRule,
    MarketQuote,
    QuotingConvention,
    PayerReceiver,
    OptionStyle,
    CDXOptionModelType,
    CDXOptionModelParam,
    CDXOptionModelData,
)
from Quantifi.Toolkit.Products import CDXOption
from Quantifi.Toolkit.Products.StandardProductTerms import CreditIndexTerms
from Quantifi.Toolkit.Curves import DiscountCurve
from Quantifi.Toolkit.Pricers import CreditIndexOptionPricer, CreditIndexOptionPricerFactory
from Quantifi.Toolkit.Calibrators.Volatilities import CalibratedVolatilitySurface
from Quantifi.Toolkit.Sensitivity import Sensitivities

pricing_date = Dt(20260915)          # today

# IG46 = CDX.NA.IG Series 46, standard 5Y underlying index terms
ig46_cdx = CreditIndexTerms.GetCDX("CDX.NA.IG", 46, 5)
ig46_cdx

<CLR: CDX.NA.IG.46 5 Years>

In [2]:
# --- Pull SOFR OIS quotes from Bloomberg -------------------------------------------
# ASSUMPTION: these are the standard USD SOFR OIS swap tickers (letter-coded for
# sub-1Y stubs, numeric for whole years) plus the published O/N SOFR fixing.
# Please sanity check these resolve on your terminal before trusting the curve -
# bdp_mm() returns 'FieldNotFound' for anything that doesn't.

from bbg_utils import bdp_mm

sofr_ticker_meta = {
    "SOFRRATE Index": ("1D", "MM"),     # O/N SOFR fixing
    "USOSFRC Curncy": ("3M", "Swap"),
    "USOSFRF Curncy": ("6M", "Swap"),
    "USOSFR1 Curncy": ("1Y", "Swap"),
    "USOSFR2 Curncy": ("2Y", "Swap"),
    "USOSFR3 Curncy": ("3Y", "Swap"),
    "USOSFR5 Curncy": ("5Y", "Swap"),
    "USOSFR7 Curncy": ("7Y", "Swap"),
}

sofr_quotes_df = bdp_mm(list(sofr_ticker_meta.keys()), "PX_LAST")
sofr_quotes_df["Tenor"] = sofr_quotes_df["security"].map(lambda t: sofr_ticker_meta[t][0])
sofr_quotes_df["InstType"] = sofr_quotes_df["security"].map(lambda t: sofr_ticker_meta[t][1])
sofr_quotes_df

,security,PX_LAST,Tenor,InstType
0,SOFRRATE Index,3.85000,1D,MM
1,USOSFR1 Curncy,4.49640,1Y,Swap
2,USOSFR2 Curncy,4.62541,2Y,Swap
3,USOSFR3 Curncy,4.61882,3Y,Swap
4,USOSFR5 Curncy,4.56880,5Y,Swap
5,USOSFR7 Curncy,4.55735,7Y,Swap
6,USOSFRC Curncy,4.03603,3M,Swap
7,USOSFRF Curncy,4.20885,6M,Swap


In [3]:
# --- Build the SOFR discount curve in Quantifi -------------------------------------
# Same MultiRateCurveFitCalibrator pattern Quantifi's own Curve Calibration sample
# uses for RFR (SOFR/ESTR/SONIA/...) discount curves.

from Quantifi.Toolkit.Base.ReferenceRates import InterestReferenceRate
from Quantifi.Toolkit.Curves import CurveTenorFactory
from Quantifi.Toolkit.Calibrators import CalibratorSettings, MultiRateCurveFitCalibrator
from Quantifi.Toolkit.Numerics import InterpScheme

sofr_ref_rate = InterestReferenceRate.Get("SOFR")

sofr_market_quotes = CurveTenorFactory.BuildTenors(
    tenorNames=sofr_quotes_df["Tenor"].tolist(),
    instruments=sofr_quotes_df["InstType"].tolist(),
    quoteValues=[[q / 100.0] for q in sofr_quotes_df["PX_LAST"].tolist()],
    quotingConventions=None,
    referenceRates=[sofr_ref_rate],
    indexTenors=["1D"],
)

sofr_calibrator_settings = CalibratorSettings()
sofr_calibrator_settings.CurveAsOf = pricing_date
sofr_calibrator_settings.ApproximateRateProjection = True
sofr_calibrator_settings.ChainedSwapApproach = True
sofr_calibrator_settings.OverlapTreatmentOrder = ["Swap", "MM"]
sofr_calibrator_settings.InterpScheme = InterpScheme.FromString("WeightedTensionC2;Smooth")

sofr_curve = MultiRateCurveFitCalibrator.FitQuotes(
    curveName="SOFR_Curve",
    category="",
    tradeDate=pricing_date,
    targetRate=sofr_ref_rate,
    targetIndexTenor="1D",
    longTargetIndexTenor=Tenor.Empty,
    basisRate=None,
    basisIndexTenor=Tenor.Empty,
    longBasisIndexTenor=Tenor.Empty,
    futuresTenor=Tenor.Empty,
    tenorQuotes=sofr_market_quotes,
    fitSettings=sofr_calibrator_settings,
    referenceCurves=[],
)

sofr_curve

<CLR: SOFR_Curve[MultiRateCurveFitCalibrator]
 name_ = SOFR_Curve
 category_ = 
 asOf_ = 15-Sep-2026
 spread_ = 0
 dc_ = Actual/365 Fixed
 freq_ = Continuous
 ccy_ = USD
 interp_ = Quantifi.Toolkit.Numerics.WeightedAdapter
 flags_ = 0
 jumpDate_ = <null>
 dt_[0] = 16-Sep-2026 ; y_[0] = 0.99989307
 dt_[1] = 17-Dec-2026 ; y_[1] = 0.98968905
 dt_[2] = 17-Mar-2027 ; y_[2] = 0.97906783
 dt_[3] = 17-Sep-2027 ; y_[3] = 0.95619457
 dt_[4] = 18-Sep-2028 ; y_[4] = 0.9119429
 dt_[5] = 17-Sep-2029 ; y_[5] = 0.87137366
 dt_[6] = 17-Sep-2031 ; y_[6] = 0.7972304
 dt_[7] = 19-Sep-2033 ; y_[7] = 0.72857016
tenors_.Count = 8; Tenor list:
tenors_[0]: Name = SOFR.MM_1D; Quote = 0.0385, Yield; weight_ = 1
tenors_[1]: Name = SOFR_1D.Swap_3M; Quote = 0.0403603, Yield; weight_ = 1
tenors_[2]: Name = SOFR_1D.Swap_6M; Quote = 0.0420885, Yield; weight_ = 1
tenors_[3]: Name = SOFR_1D.Swap_1Y; Quote = 0.044964, Yield; weight_ = 1
tenors_[4]: Name = SOFR_1D.Swap_2Y; Quote = 0.0462541, Yield; weight_ = 1
tenors_[5]:

In [4]:
# --- Basic single-option pricing: IG46 December 55 payer, 37 vol -----------------

# INPUTS you can change
ig46_spot_bp = 52.0                  # <-- set the IG46 spread level here
strike_bp = 55.0
flat_vol = 0.426

discount_curve = sofr_curve

# December expiry: 3rd-Wednesday IMM roll in the month ~3M out from pricing date.
# ASSUMPTION: adjust this explicitly if the actual listed IG46 Dec option expiry differs.
option_expiry = (pricing_date + Tenor("3M")).ApplyCycleRule(CycleRule.IMM)

# Standard CDX Option model settings (matches Quantifi's own CDX Options sample):
# full-spread model, spot spread adjusted to a forward via the market method.
model_data = CDXOptionModelData()
model_data.Choice = (
    CDXOptionModelParam.AdjustSpread
    | CDXOptionModelParam.AdjustSpreadByMarketMethod
    | CDXOptionModelParam.ForceFlatMarketCurve
    | CDXOptionModelParam.ForceCleanValue
    | CDXOptionModelParam.MarketPayoff
    | CDXOptionModelParam.HandleIndexFactors
)

vol_surface = CalibratedVolatilitySurface.FromFlatVolatility(pricing_date, flat_vol)

spot_quote = MarketQuote(ig46_spot_bp / 10000.0, ig46_cdx.QuotingConvention)

ig46_dec_payer = CDXOption(
    effective=ig46_cdx.Effective,
    ccy=ig46_cdx.Ccy,
    underlying=ig46_cdx,
    expiration=option_expiry,
    type=PayerReceiver.Payer,
    style=OptionStyle.European,
    strike=strike_bp / 10000.0,
    strikeIsPrice=False,
    indexFactor=1.0,
)

pricer = CreditIndexOptionPricer(
    ig46_dec_payer,
    pricing_date,
    Dt.AddDays(pricing_date, 1, None),   # settle date
    discount_curve,
    spot_quote,
    None,                                  # cdxSettleDate
    ig46_cdx.Recovery,
    0,                                      # basketSize (0 -> infer from CDX)
    None,                                   # survivalCurves (single-name CDS equivalent approx)
    1.0,                                    # currentFactor
    float("nan"),                           # existingLossToInclude
    CDXOptionModelType.FullSpread,
    model_data,
    vol_surface,
)
pricer.Notional = 10_000_000.0

# --- results ---
forward_spread_bp = CreditIndexOptionPricerFactory.EffectiveForwardSpread(pricer) * 10000
fair_price = CreditIndexOptionPricerFactory.FairPrice(pricer)
premium_bp = CreditIndexOptionPricerFactory.OptionPremium(pricer) * 10000
delta = CreditIndexOptionPricerFactory.MarketDelta(pricer)
gamma = CreditIndexOptionPricerFactory.MarketGamma(pricer)
vega = CreditIndexOptionPricerFactory.Vega(pricer)
theta = CreditIndexOptionPricerFactory.MarketTheta(pricer)

print(f"Expiry date:            {option_expiry}")
print(f"Spot IG46 spread:       {ig46_spot_bp:.2f} bp")
print(f"Implied Dec fwd spread: {forward_spread_bp:.2f} bp  (verify vs ~55bp expectation)")
print(f"Recovery assumed:       {ig46_cdx.Recovery:.2%}")
print()
print(f"Fair price (per unit):  {fair_price:.6f}")
print(f"PV (on notional):       {fair_price * pricer.Notional:,.0f}")
print(f"Premium:                {premium_bp:.2f} cts")
print(f"Delta:                  {delta:.4f}")
print(f"Gamma:                  {gamma:.4f}")
print(f"Vega:                   {vega:.4f}")
print(f"Theta:                  {theta:.4f}")

Expiry date:            16-Dec-2026
Spot IG46 spread:       52.00 bp
Implied Dec fwd spread: 55.38 bp  (verify vs ~55bp expectation)
Recovery assumed:       40.00%

Fair price (per unit):  0.001969
PV (on notional):       19,693
Premium:                19.69 cts
Delta:                  0.5695
Gamma:                  0.2696
Vega:                   444.2089
Theta:                  -181.7918
